# 7. Hybrid with 0.55 cutoff

User-selected non-LLM hybrid reference; not a retrieval win over improved BM25.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(7)
model['config_snapshot']

{'name': 'hybrid_bm25_minilm_rrf_abstention',
 'retriever_type': 'hybrid',
 'bm25_config': 'configs/bm25_focused_stemmed.json',
 'dense_config': 'semantic_configs/minilm.json',
 'candidate_pool': 20,
 'rrf_constant': 60,
 'bm25_weight': 0.75,
 'dense_confidence_threshold': 0.55}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.144336,0.319079,0.435714
1,3,140,0.091631,0.534831,0.650000
2,5,140,0.071166,0.655133,0.750000
3,10,140,0.045645,0.803197,0.871429


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 9}
False answerable abstention: 0.007142857142857143
Cached latency: {'median': 10.737449978478253, 'p95': 13.93350497819483}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
5 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
2,cuad/HOLIDAYRVSUPERSTORESINC_04_15_2002-EX-10....,4800,5800,"he License (the ""Endorsement"").\n\n 3. ..."
3,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
4,"cuad/Loop Industries, Inc. - Marketing Agreeme...",19200,20200,"greement; or (vii) in confidence, in connectio..."


Full code walkthrough: ../experiments/17_hybrid_bm25_minilm.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.